**Set Up**

In [0]:
%sql
-- Owner: Nadine
-- Name: 01 - Setup
-- Purpose: Create the Bronze, Silver, and Gold schemas and validate that the source folder contains exactly the six approved Instacart CSV files.
-- Grain: One validation summary row for the Instacart source folder.

CREATE SCHEMA IF NOT EXISTS workspace.instacart_bronze;
CREATE SCHEMA IF NOT EXISTS workspace.instacart_silver;
CREATE SCHEMA IF NOT EXISTS workspace.instacart_gold;

WITH expected_files AS (
  SELECT explode(
    array(
      'aisles.csv',
      'departments.csv',
      'order_products__prior.csv',
      'order_products__train.csv',
      'orders.csv',
      'products.csv'
    )
  ) AS file_name
),
actual_files AS (
  SELECT
    regexp_extract(path, '([^/]+)$', 1) AS file_name
  FROM read_files(
    '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/*.csv',
    format => 'binaryFile'
  )
),
file_checks AS (
  SELECT
    (SELECT COUNT(*) FROM actual_files) AS actual_file_count,
    (
      SELECT COUNT(*)
      FROM actual_files a
      LEFT ANTI JOIN expected_files e
        ON a.file_name = e.file_name
    ) AS unexpected_file_count,
    (
      SELECT COUNT(*)
      FROM expected_files e
      LEFT ANTI JOIN actual_files a
        ON e.file_name = a.file_name
    ) AS missing_file_count
)
SELECT
  actual_file_count,
  unexpected_file_count,
  missing_file_count,
  assert_true(
    actual_file_count = 6,
    'source folder must contain exactly six csv files'
  ) AS file_count_check,
  assert_true(
    unexpected_file_count = 0,
    'source folder contains an unexpected csv file'
  ) AS unexpected_file_check,
  assert_true(
    missing_file_count = 0,
    'one or more required source files are missing'
  ) AS missing_file_check
FROM file_checks;

**Bronze per Table**

In [0]:
%sql
-- Owner: Nadine
-- Name: 02 - Bronze Aisles
-- Purpose: Load Instacart aisle records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per aisle, uniquely identified by aisle_id.

USE CATALOG workspace;
CREATE SCHEMA IF NOT EXISTS instacart_bronze;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE aisles
USING DELTA -- create this table as a Delta Lake table to add features on top of ordinary files, such as ACID transactions
COMMENT 'bronze copy of the instacart aisles csv' -- adds a description to the table as metadata.
AS
SELECT
  aisle_id,
  aisle,
  _rescued_data -- to capture source data that does not fit the schema specified
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/aisles.csv',
  format => 'csv',
  header => true,
  schema => 'aisle_id INT, aisle STRING',
  rescuedDataColumn => '_rescued_data'
);

In [0]:
%sql
-- Owner: Nadine
-- Name: 03 - Bronze Departments
-- Purpose: Load Instacart department records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per department, uniquely identified by department_id.

USE CATALOG workspace;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE departments 
USING DELTA
COMMENT 'bronze copy of the instacart departments csv'
AS
SELECT
  department_id,
  department,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/departments.csv',
  format => 'csv',
  header => true,
  schema => 'department_id INT, department STRING',
  rescuedDataColumn => '_rescued_data'
);

In [0]:
%sql
-- Owner: Nadine
-- Name: 04 - Bronze Products
-- Purpose: Load Instacart product records into the Bronze Delta table using an explicit schema and correct quotation-mark parsing while retaining rescued data.
-- Grain: One row per product, uniquely identified by product_id.

USE CATALOG workspace;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE workspace.instacart_bronze.products
USING DELTA
COMMENT 'bronze copy of the instacart products csv'
AS
SELECT
  product_id,
  product_name,
  aisle_id,
  department_id,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/products.csv',
  format => 'csv',
  header => true,
  quote => '"',
  escape => '"',
  schema => 'product_id INT, product_name STRING, aisle_id INT, department_id INT',
  rescuedDataColumn => '_rescued_data'
);

In [0]:
%sql
-- Owner: Nadine
-- Name: 05 - Bronze Orders
-- Purpose: Load Instacart order records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per Instacart order, uniquely identified by order_id.

USE CATALOG workspace;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE orders 
USING DELTA
COMMENT 'bronze copy of the instacart orders csv'
AS
SELECT
  order_id,
  user_id,
  eval_set,
  order_number,
  order_dow,
  order_hour_of_day,
  days_since_prior_order,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/orders.csv',
  format => 'csv',
  header => true,
  schema => 'order_id INT, user_id INT, eval_set STRING, order_number INT, order_dow INT, order_hour_of_day INT, days_since_prior_order DOUBLE',
  rescuedDataColumn => '_rescued_data'
);

In [0]:
%sql
-- Owner: Nadine
-- Name: 06 - Bronze Order Products Prior
-- Purpose: Load prior order-product CSV records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per product line in one prior order, uniquely identified by (order_id, add_to_cart_order).

USE CATALOG workspace;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE order_products_prior 
USING DELTA
COMMENT 'bronze copy of the instacart prior order products csv'
AS
SELECT
  order_id,
  product_id,
  add_to_cart_order,
  reordered,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/order_products__prior.csv',
  format => 'csv',
  header => true,
  schema => 'order_id INT, product_id INT, add_to_cart_order INT, reordered INT',
  rescuedDataColumn => '_rescued_data'
);

In [0]:
%sql
-- Owner: Nadine
-- Name: 07 - Bronze Order Products Train
-- Purpose: Load train order-product records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per product line in one train order, uniquely identified by (order_id, add_to_cart_order).

USE CATALOG workspace;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE order_products_train 
USING DELTA
COMMENT 'bronze copy of the instacart train order products csv'
AS
SELECT
  order_id,
  product_id,
  add_to_cart_order,
  reordered,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/order_products__train.csv',
  format => 'csv',
  header => true,
  schema => 'order_id INT, product_id INT, add_to_cart_order INT, reordered INT',
  rescuedDataColumn => '_rescued_data'
);

**Bronze Validation**

In [0]:
%sql
-- Owner: Nadine
-- Name: 08 - Bronze Validation
-- Purpose: Validate Bronze row counts, required identifiers, candidate keys, required fields, domain rules, and rescued data with table-level pass or fail results.
-- Grain: One validation summary row per Bronze table.

WITH validation AS (
  SELECT
    'aisles' AS table_name,
    134 AS expected_rows,
    COUNT(*) AS actual_rows,
    COUNT(*) - 134 AS row_difference,
    COUNT_IF(aisle_id IS NULL) AS null_required_ids,
    COUNT(*) - COUNT(DISTINCT aisle_id) AS duplicate_primary_keys,
    0 AS duplicate_alternate_keys,
    COUNT_IF(aisle IS NULL OR TRIM(aisle) = '') AS required_field_issues,
    0 AS domain_issues,
    COUNT_IF(_rescued_data IS NOT NULL) AS rescued_rows
  FROM workspace.instacart_bronze.aisles

  UNION ALL

  SELECT
    'departments',
    21,
    COUNT(*),
    COUNT(*) - 21,
    COUNT_IF(department_id IS NULL),
    COUNT(*) - COUNT(DISTINCT department_id),
    0,
    COUNT_IF(department IS NULL OR TRIM(department) = ''),
    0,
    COUNT_IF(_rescued_data IS NOT NULL)
  FROM workspace.instacart_bronze.departments

  UNION ALL

  SELECT
    'products',
    49688,
    COUNT(*),
    COUNT(*) - 49688,
    COUNT_IF(
      product_id IS NULL
      OR aisle_id IS NULL
      OR department_id IS NULL
    ),
    COUNT(*) - COUNT(DISTINCT product_id),
    0,
    COUNT_IF(product_name IS NULL OR TRIM(product_name) = ''),
    0,
    COUNT_IF(_rescued_data IS NOT NULL)
  FROM workspace.instacart_bronze.products

  UNION ALL

  SELECT
    'orders',
    3421083,
    COUNT(*),
    COUNT(*) - 3421083,
    COUNT_IF(order_id IS NULL OR user_id IS NULL),
    COUNT(*) - COUNT(DISTINCT order_id),
    COUNT(*) - COUNT(DISTINCT STRUCT(user_id, order_number)),
    COUNT_IF(
      eval_set IS NULL
      OR TRIM(eval_set) = ''
      OR order_number IS NULL
      OR order_dow IS NULL
      OR order_hour_of_day IS NULL
    ),
    COUNT_IF(
      eval_set NOT IN ('prior', 'train', 'test')
      OR order_number < 1
      OR order_dow NOT BETWEEN 0 AND 6
      OR order_hour_of_day NOT BETWEEN 0 AND 23
      OR days_since_prior_order < 0
      OR (order_number = 1 AND days_since_prior_order IS NOT NULL)
      OR (order_number > 1 AND days_since_prior_order IS NULL)
    ),
    COUNT_IF(_rescued_data IS NOT NULL)
  FROM workspace.instacart_bronze.orders

  UNION ALL

  SELECT
    'order_products_prior',
    32434489,
    COUNT(*),
    COUNT(*) - 32434489,
    COUNT_IF(
      order_id IS NULL
      OR product_id IS NULL
      OR add_to_cart_order IS NULL
    ),
    COUNT(*) - COUNT(DISTINCT STRUCT(order_id, add_to_cart_order)),
    COUNT(*) - COUNT(DISTINCT STRUCT(order_id, product_id)),
    0,
    COUNT_IF(
      add_to_cart_order < 1
      OR reordered IS NULL
      OR reordered NOT IN (0, 1)
    ),
    COUNT_IF(_rescued_data IS NOT NULL)
  FROM workspace.instacart_bronze.order_products_prior

  UNION ALL

  SELECT
    'order_products_train',
    1384617,
    COUNT(*),
    COUNT(*) - 1384617,
    COUNT_IF(
      order_id IS NULL
      OR product_id IS NULL
      OR add_to_cart_order IS NULL
    ),
    COUNT(*) - COUNT(DISTINCT STRUCT(order_id, add_to_cart_order)),
    COUNT(*) - COUNT(DISTINCT STRUCT(order_id, product_id)),
    0,
    COUNT_IF(
      add_to_cart_order < 1
      OR reordered IS NULL
      OR reordered NOT IN (0, 1)
    ),
    COUNT_IF(_rescued_data IS NOT NULL)
  FROM workspace.instacart_bronze.order_products_train
),
results AS (
  SELECT
    table_name,
    expected_rows,
    actual_rows,
    row_difference,
    null_required_ids,
    duplicate_primary_keys,
    duplicate_alternate_keys,
    required_field_issues,
    domain_issues,
    rescued_rows,
    CASE
      WHEN row_difference = 0
        AND null_required_ids = 0
        AND duplicate_primary_keys = 0
        AND duplicate_alternate_keys = 0
        AND required_field_issues = 0
        AND domain_issues = 0
        AND rescued_rows = 0
      THEN 'PASS'
      ELSE 'FAIL'
    END AS status
  FROM validation
)
SELECT
  table_name,
  expected_rows,
  actual_rows,
  row_difference,
  null_required_ids,
  duplicate_primary_keys,
  duplicate_alternate_keys,
  required_field_issues,
  domain_issues,
  rescued_rows,
  status,
  assert_true(
    SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) OVER () = 0,
    'one or more bronze tables failed validation; review the table-level metrics'
  ) AS bronze_validation_check
FROM results
ORDER BY table_name;
